In [47]:
import polars as pl

In [48]:
#Load df
df = pl.read_csv("../data/output/v2/2.clean_df_v2.csv")

df = df.rename({
    "Start date": "start_date",
    "Start station number": "start_station",
    "End date": "end_date",
    "End station number": "end_station",
    "Total duration (ms)": "duration_ms",
    "Operating date": "operating_date",
    "Day of week": "day_of_week",
    "Hour of day": "hour"
})

df = df.with_columns([
    pl.col("start_date").str.to_datetime(strict=False),
    pl.col("end_date").str.to_datetime(strict=False),
    pl.col("operating_date").str.to_date(strict=False),
])
df

start_date,start_station,end_date,end_station,duration_ms,operating_date,day_of_week,hour,start_station_latitude,start_station_longitude,start_station_name,end_station_latitude,end_station_longitude,end_station_name
datetime[μs],i64,datetime[μs],i64,i64,date,i64,i64,f64,f64,str,f64,f64,str
2025-01-14 23:59:00,1043,2025-01-15 00:13:00,200149,870468,2025-01-14,2,23,51.517821,-0.096496,"""Museum of London, Barbican""",51.511542,-0.056667,"""Watney Street, Shadwell"""
2025-01-14 23:59:00,300015,2025-01-15 00:04:00,300229,307181,2025-01-14,2,23,51.472509,-0.122831,"""Binfield Road, Stockwell""",51.469202,-0.119022,"""Sidney Road, Stockwell"""
2025-01-14 23:59:00,1068,2025-01-15 00:10:00,1051,640755,2025-01-14,2,23,51.521113,-0.078869,"""Norton Folgate, Liverpool Stre…",51.534042,-0.086379,"""Shoreditch Park, Hoxton"""
2025-01-14 23:59:00,1159,2025-01-15 00:10:00,1007,642548,2025-01-14,2,23,51.522853,-0.099994,"""Berry Street, Clerkenwell""",51.51477,-0.122219,"""Drury Lane, Covent Garden"""
2025-01-14 23:58:00,200048,2025-01-15 00:14:00,1058,944185,2025-01-14,2,23,51.493978,-0.127554,"""Page Street, Westminster""",51.510212,0.004979,"""Orchard Place, Blackwall Tunne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-16 00:02:00,200141,2025-12-16 00:16:00,200237,837578,2025-12-16,2,0,51.509158,-0.224103,"""Westfield Ariel Way, White Cit…",51.472817,-0.199783,"""Parson's Green , Parson's Gree…"
2025-12-16 00:01:00,200011,2025-12-16 00:16:00,200250,908383,2025-12-16,2,0,51.519265,-0.021345,"""Furze Green, Bow""",51.520893,-0.051394,"""Cleveland Way, Stepney"""
2025-12-16 00:00:00,300096,2025-12-16 00:07:00,962,432164,2025-12-16,2,0,51.511891,-0.107349,"""Tallis Street, Temple""",51.505569,-0.111606,"""Stamford Street, South Bank"""


In [49]:
#########################
#Create station profiles
#########################

#-------------------
#departure features
#-------------------
departures = (
    df
    .group_by("start_station")
    .agg([
        pl.len().alias("n_departures"), #number of trips starting at this station

        pl.col("duration_ms").mean().alias("mean_duration"), #average trip duration for trips departing from that station.
        pl.col("duration_ms").median().alias("median_duration"), #median trip duration departing from that station.
        pl.col("duration_ms").std().alias("std_duration"), #standard deviation trip duration departing from that station.

        pl.col("end_station").n_unique().alias("n_destinations"), #Counts the number of different destination stations reached from each starting station.
    ])
    .rename({"start_station": "station"})
)


#--------------------
#arrival features
#--------------------
arrivals = (
    df
    .group_by("end_station")
    .agg([
        pl.len().alias("n_arrivals"), #number of trips arriving at this station

        pl.col("start_station").n_unique().alias("n_origins"), #Counts the number of different destination stations that end in each end station.
    ])
    .rename({"end_station": "station"})
)


#Join them
stations = departures.join(
    arrivals,
    on="station",
    how="full"
).fill_null(0)

stations

station,n_departures,mean_duration,median_duration,std_duration,n_destinations,station_right,n_arrivals,n_origins
i64,u32,f64,f64,f64,u32,i64,u32,u32
10632,4058,1.2457e6,880538.5,4.2380e6,452,10632,4031,439
22169,11770,905153.138233,542541.0,1.0419e7,615,22169,11779,618
200053,3280,2.1102e6,925209.5,4.0010e7,466,200053,3097,426
300048,7946,1.0453e6,742999.5,5.7007e6,374,300048,7796,409
22175,6259,1.5980e6,889888.0,2.1288e7,393,22175,6160,406
…,…,…,…,…,…,…,…,…
1045,9855,1.2386e6,761825.0,2.0698e7,655,1045,9912,645
1179,7539,1.0254e6,763257.0,3.7795e6,542,1179,7534,533
200157,14369,1.0581e6,743040.0,4.2008e6,646,200157,14398,661


In [50]:
#-------------------
#departure/arrival ratio
#-------------------

stations = stations.with_columns(
    (pl.col('n_departures')/pl.col('n_arrivals')).alias('departure_arrival_ratio')
)

stations
    

station,n_departures,mean_duration,median_duration,std_duration,n_destinations,station_right,n_arrivals,n_origins,departure_arrival_ratio
i64,u32,f64,f64,f64,u32,i64,u32,u32,f64
10632,4058,1.2457e6,880538.5,4.2380e6,452,10632,4031,439,1.006698
22169,11770,905153.138233,542541.0,1.0419e7,615,22169,11779,618,0.999236
200053,3280,2.1102e6,925209.5,4.0010e7,466,200053,3097,426,1.059089
300048,7946,1.0453e6,742999.5,5.7007e6,374,300048,7796,409,1.019241
22175,6259,1.5980e6,889888.0,2.1288e7,393,22175,6160,406,1.016071
…,…,…,…,…,…,…,…,…,…
1045,9855,1.2386e6,761825.0,2.0698e7,655,1045,9912,645,0.994249
1179,7539,1.0254e6,763257.0,3.7795e6,542,1179,7534,533,1.000664
200157,14369,1.0581e6,743040.0,4.2008e6,646,200157,14398,661,0.997986


In [51]:
#-------------------
#Avg departures and arrivals per day
#-------------------
daily_departures = (
    df
    .group_by(["operating_date", "start_station"])
    .agg(
        pl.len().alias("departures")
    )
)

daily_arrivals = (
    df
    .group_by(["operating_date", "end_station"])
    .agg(
        pl.len().alias("arrivals")
    )
)

avg_departures = (
    daily_departures
    .group_by("start_station")
    .agg(
        pl.col("departures").mean().alias("avg_daily_departures")
    )
    .rename({"start_station": "station"})
)

avg_arrivals = (
    daily_arrivals
    .group_by("end_station")
    .agg(
        pl.col("arrivals").mean().alias("avg_daily_arrivals")
    )
    .rename({"end_station": "station"})
)

station_daily_avg = (
    avg_departures
    .join(avg_arrivals, on="station", how="left")
    .sort("station")
)

station_daily_avg

station,avg_daily_departures,avg_daily_arrivals
i64,f64,f64
959,25.78022,26.406593
960,100.381868,121.17033
961,28.717033,28.89011
962,30.870523,32.909091
963,33.346154,38.635616
…,…,…
200191444,9.115385,9.192308
200230444,58.301075,56.308511
300032444,26.478261,25.73913


In [52]:
#----------------
#add departures per hour
#----------------

# hour_profile = (
#     df
#     .group_by(["start_station", "hour"])
#     .agg(
#         pl.len().alias("count")
#     )
# )


# hour_profile = (
#     hour_profile
#     .pivot(
#         on="hour",
#         index="start_station",
#         values="count"
#     )
#     .fill_null(0)
# )


# hour_profile = hour_profile.rename({
#     "start_station": "station"
# })

# hour_profile

In [53]:
#--------------------
# Normalize hourly profile
#--------------------

# hour_columns = [str(i) for i in range(24) if str(i) in hour_profile.columns]

# hour_profile = hour_profile.with_columns(
#     [
#         (pl.col(c) / pl.sum_horizontal(hour_columns)).alias(f"{c}_hours")
#         for c in hour_columns
#     ]
# ).drop(hour_columns)

# hour_profile

In [54]:
#-----------------------
#Weekend and weekday behavior
#-----------------------
print(
    df
    .select("day_of_week")
    .unique()
    .sort("day_of_week")
)

df = df.with_columns(
    (pl.col("day_of_week") >= 6).alias("is_weekend")
)

print(df)

weekend_profile = (
    df
    .group_by("start_station")
    .agg([
        pl.col("is_weekend").mean().alias("weekend_ratio")
    ])
    .rename({"start_station": "station"})
)

weekend_profile

shape: (7, 1)
┌─────────────┐
│ day_of_week │
│ ---         │
│ i64         │
╞═════════════╡
│ 1           │
│ 2           │
│ 3           │
│ 4           │
│ 5           │
│ 6           │
│ 7           │
└─────────────┘
shape: (8_762_355, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ start_dat ┆ start_sta ┆ end_date  ┆ end_stati ┆ … ┆ end_stati ┆ end_stati ┆ end_stati ┆ is_weeke │
│ e         ┆ tion      ┆ ---       ┆ on        ┆   ┆ on_latitu ┆ on_longit ┆ on_name   ┆ nd       │
│ ---       ┆ ---       ┆ datetime[ ┆ ---       ┆   ┆ de        ┆ ude       ┆ ---       ┆ ---      │
│ datetime[ ┆ i64       ┆ μs]       ┆ i64       ┆   ┆ ---       ┆ ---       ┆ str       ┆ bool     │
│ μs]       ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2025-01-1 ┆ 1043      ┆ 2025-01-1 ┆ 200149    

station,weekend_ratio
i64,f64
300042,0.239737
1054,0.199424
200047,0.2719
300200,0.35973
300054,0.25753
…,…
200014,0.327008
200133,0.26515
200136,0.143114


In [55]:
#--------------------
#Rush hour features
#--------------------

df = df.with_columns([
    (
        (pl.col("hour") >= 7) &
        (pl.col("hour") <= 9)
    ).alias("morning_rush"),

    (
        (pl.col("hour") >= 17) &
        (pl.col("hour") <= 19)
    ).alias("evening_rush"),

    (
        (pl.col("hour") >= 0) &
        (pl.col("hour") <= 5)
    ).alias("night")
])

rush_profile = (
    df
    .group_by("start_station")
    .agg([
        pl.col("morning_rush").mean().alias("morning_rush_ratio"),
        pl.col("evening_rush").mean().alias("evening_rush_ratio"),
        pl.col("night").mean().alias("night_ratio"),
    ])
    .rename({"start_station": "station"})
)

rush_profile

station,morning_rush_ratio,evening_rush_ratio,night_ratio
i64,f64,f64,f64
300060,0.306729,0.189456,0.02856
300042,0.275862,0.227217,0.022373
1069,0.151697,0.280297,0.037639
300072,0.308251,0.181848,0.016667
1057,0.31608,0.186008,0.016706
…,…,…,…
1167,0.319267,0.274001,0.007213
200148,0.208458,0.201137,0.027079
200008,0.080437,0.392508,0.013632


In [56]:
#-------------------
#Final df -> stations
#-------------------

stations = (
    stations
    #.join(hour_profile, on="station", how="left")
    .join(station_daily_avg, on="station", how="left")
    .join(weekend_profile, on="station", how="left")
    .join(rush_profile, on="station", how="left")
    .fill_null(0)
)

print(stations.shape)


(822, 16)


In [57]:
stations = stations.select(['station','n_departures','avg_daily_departures','n_arrivals','avg_daily_arrivals','departure_arrival_ratio','n_destinations','n_origins','mean_duration','median_duration','std_duration','weekend_ratio','morning_rush_ratio','evening_rush_ratio','night_ratio'])

stations

station,n_departures,avg_daily_departures,n_arrivals,avg_daily_arrivals,departure_arrival_ratio,n_destinations,n_origins,mean_duration,median_duration,std_duration,weekend_ratio,morning_rush_ratio,evening_rush_ratio,night_ratio
i64,u32,f64,u32,f64,f64,u32,u32,f64,f64,f64,f64,f64,f64,f64
10632,4058,11.209945,4031,11.197222,1.006698,452,439,1.2457e6,880538.5,4.2380e6,0.174963,0.126417,0.245195,0.014539
22169,11770,32.335165,11779,32.35989,0.999236,615,618,905153.138233,542541.0,1.0419e7,0.23237,0.223874,0.223959,0.026423
200053,3280,9.111111,3097,8.53168,1.059089,466,426,2.1102e6,925209.5,4.0010e7,0.21372,0.217378,0.283841,0.031707
300048,7946,22.383099,7796,21.898876,1.019241,374,409,1.0453e6,742999.5,5.7007e6,0.261515,0.40813,0.158067,0.017871
22175,6259,17.195055,6160,16.876712,1.016071,393,406,1.5980e6,889888.0,2.1288e7,0.281195,0.275763,0.150344,0.044736
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1045,9855,27.14876,9912,27.230769,0.994249,655,645,1.2386e6,761825.0,2.0698e7,0.121157,0.149467,0.41451,0.017453
1179,7539,20.654795,7534,20.697802,1.000664,542,533,1.0254e6,763257.0,3.7795e6,0.233055,0.21024,0.267277,0.0256
200157,14369,39.475275,14398,39.446575,0.997986,646,661,1.0581e6,743040.0,4.2008e6,0.314149,0.065419,0.308999,0.022131


In [59]:
#######################
#export df
#######################

stations.write_csv("../data/output/v2/4.stations_df.csv")
